# Debug access rigths harvesting


#### Imports

In [7]:
import fire
import re
import gspread
import random
import json
from collections import Counter
import os

### Code

## 1. Adapt the Gsheet fetcher to the access-rights case

In [48]:
spreadsheet_id = '1cLgvhFTPaqjnxTByDejMrwxBM0-o0j1U2Oz-o4oTabs'
worksheet_name = 'v5'
credentials_path = '/Users/piconti/impresso/impresso-corpus-metadata/credentials.json'
output_file = '/Users/piconti/impresso/impresso-corpus-metadata/access_rights/test_values.json'

In [49]:
# Initialize gspread client
gc = gspread.service_account(filename=credentials_path)

In [50]:
# Open spreadsheet using URL
sheet = gc.open_by_key(spreadsheet_id)
worksheet_list = sheet.worksheets()

if worksheet_name not in [worksheet.title for worksheet in worksheet_list]:
    print(
        f"Worksheet {worksheet_name} not found in the spreadsheet. Available sheets:"
    )
    print(worksheet_list)

# Open worksheet by name
worksheet = sheet.worksheet(worksheet_name)
worksheet

<Worksheet 'v5' id:209232756>

In [51]:
# Get worksheet data
values = worksheet.get_all_records(2)
values

[{'': 1,
  'Alias': 'NGV',
  'Title': 'La nouvelle Gazette de Vaud',
  'Included Time Period\nSTART DATE (1st January)\n\n(to be filled only if it applies)': 1865,
  'Included Time Period\nEND DATE (31st December)\n\n(to be filled only if it applies)': 1904,
  'Do you provide only metadata (and not also content)?': 'no',
  "What is the copyright status of this title for the given time period?\nIf 'Public domain', no need to fill in the other columns whose values are then understood as 'Yes'.": 'Public Domain',
  'To which registered user statuses do you want to restrict the EXPLORE action on this title?\n\n(sufficient condition)': 'No restriction (all registered users allowed)',
  'To which registered user statuses do you want to restrict the GET action on TRANSCRIPTS of this title?\n\n(sufficient condition)': 'No restriction (all registered users allowed)',
  'To which registered user statuses do you want to restrict the GET action on IMAGES | ANY PART OF THE FACSIMILE of this title?\

In [3]:
ACCESS_RIGHTS_RULES = {
    "partner_id": {"rename_key_to": "rights_holder_id"},
    "alias": {"rename_key_to": "title_alias"},
    "title": {"rename_key_to": "full_title"},
    "included_time_period_start_date_1st_january_to_be_filled_only_if_it_applies": {
        "rename_key_to": "start_year"
    },
    "included_time_period_end_date_31st_december_to_be_filled_only_if_it_applies": {
        "rename_key_to": "end_year"
    },
    "do_you_provide_content_and_not_only_metadata": {"rename_key_to": "metadata_only"},
    "what_is_the_copyright_status_of_the_content_of_this_title_for_the_given_time_period_if_public_domain_no_need_to_fill_the_columns_h_i_j_whose_values_are_then_understood_as_no_restriction": {
        "rename_key_to": "copyright_status"
    },
    "which_user_status_or_archive_membership_is_sufficient_to_execute_the_explore_action_on_this_title": {
        "rename_key_to": "explore_req_status"
    },
    "which_user_status_or_archive_membership_is_sufficient_to_execute_the_get_action_on_transcripts_of_this_title": {
        "rename_key_to": "get_transcript_req_status"
    },
    "which_user_status_or_archive_membership_is_sufficient_to_execute_thethe_get_action_on_images_any_part_of_the_facsimile_of_this_title": {
        "rename_key_to": "get_facsimile_req_status"
    },
}


def util_slugify(text: str) -> str:
    """
    Converts a string to a slug by replacing spaces with underscores and converting to lowercase.

    Args:
      text (str): Input string to slugify.

    Returns:
      str: Slugified string.
    """
    s = text.lower().strip()
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s_-]+", "_", s)
    s = re.sub(r"(^-+)|(-+$)", "", s)
    return s


def transform_value(d: dict, is_metadata: bool = True) -> dict:
    """
    Change the value of a key in a dictionary.

    Args:
      d (dict): Input dictionary.
      key (str): Key to change.
      value (str): New value for the key.

    Returns:
      dict: Dictionary with the updated key value.
    """
    transformed = {}

    for key in d.keys():
        slugified_key = util_slugify(key)
        transformed[slugified_key] = d[key]

    # depending on the data to fetch, different rules should be applied.
    rules = ACCESS_RIGHTS_RULES

    for key_with_rule, rule in rules.items():
        if "copy_value_from_field" in rule:
            transformed[key_with_rule] = transformed[rule["copy_value_from_field"]]
        if "split_values_by_re" in rule:
            transformed[key_with_rule] = [
                x
                for x in re.split(
                    rule["split_values_by_re"], transformed[key_with_rule]
                )
                if x.strip()
            ]
        if "rename_key_to" in rule:
            transformed[rule["rename_key_to"]] = transformed[key_with_rule]
            del transformed[key_with_rule]

    return transformed


In [24]:
transformed_values = list(map(transform_value, values, [False] * len(values)))
transformed_values

[{'': 1,
  'title_alias': 'NGV',
  'full_title': 'La nouvelle Gazette de Vaud',
  'start_year': 1865,
  'end_year': 1904,
  'metadata_only': 'no',
  'copyright_status': 'Public Domain ',
  'explore_status': 'No restriction (all registered users allowed)',
  'get_transcript_status': 'No restriction (all registered users allowed)',
  'get_facsimile_status': 'No restriction (all registered users allowed)',
  'status_to_validate': 'No validation'},
 {'': 2,
  'title_alias': 'NGV',
  'full_title': 'La nouvelle Gazette de Vaud',
  'start_year': 1905,
  'end_year': 1950,
  'metadata_only': 'no',
  'copyright_status': 'Copyright undetermined (Protected Domain)',
  'explore_status': 'Academic and Student statuses only',
  'get_transcript_status': 'Academic and Student statuses only',
  'get_facsimile_status': 'Academic and Student statuses only',
  'status_to_validate': 'All statuses'},
 {'': 3,
  'title_alias': 'LNC',
  'full_title': 'Le Nouveau Champion',
  'start_year': 1914,
  'end_year': 1

In [6]:
util_slugify("To be filled only if the choices made for I, J and K is Only Archive members. Which uses of the data are permitted for archive members? (permitted uses in other cases result from the user status and are defined in DSA 2.9)")

'to_be_filled_only_if_the_choices_made_for_i_j_and_k_is_only_archive_members_which_uses_of_the_data_are_permitted_for_archive_members_permitted_uses_in_other_cases_result_from_the_user_status_and_are_defined_in_dsa_29'

## 2. Convert the fetched JSON to the masterfile format

In [49]:
debug_fetched_json_path = '/Users/piconti/impresso/impresso-corpus-metadata/data/gdrive_access_rights/gsheet_access_rights.debug.json'

with open(debug_fetched_json_path, "r", encoding="utf-8") as file:
    fetched_ar = json.load(file)

fetched_ar

[{'': 1,
  'partner_id': 'Gruyere',
  'title_alias': 'NGV',
  'full_title': 'La nouvelle Gazette de Vaud',
  'start_year': 1865,
  'end_year': 1904,
  'metadata_only': 'yes',
  'copyright_status': 'Public Domain',
  'explore_req_status': 'No restriction (all registered users allowed)',
  'get_transcript_req_status': 'No restriction (all registered users allowed)',
  'get_facsimile_req_status': 'No restriction (all registered users allowed)'},
 {'': 2,
  'partner_id': 'Gruyere',
  'title_alias': 'NGV',
  'full_title': 'La nouvelle Gazette de Vaud',
  'start_year': 1905,
  'end_year': 1950,
  'metadata_only': 'yes',
  'copyright_status': 'Protected Domain: Copyright undetermined ',
  'explore_req_status': 'Educational users at least OR Archive members',
  'get_transcript_req_status': 'Educational users at least OR Archive members',
  'get_facsimile_req_status': 'Educational users at least OR Archive members'},
 {'': 3,
  'partner_id': 'LES',
  'title_alias': 'LNC',
  'full_title': 'Le No

In [50]:
first_ex = fetched_ar[0]
first_ex

{'': 1,
 'partner_id': 'Gruyere',
 'title_alias': 'NGV',
 'full_title': 'La nouvelle Gazette de Vaud',
 'start_year': 1865,
 'end_year': 1904,
 'metadata_only': 'yes',
 'copyright_status': 'Public Domain',
 'explore_req_status': 'No restriction (all registered users allowed)',
 'get_transcript_req_status': 'No restriction (all registered users allowed)',
 'get_facsimile_req_status': 'No restriction (all registered users allowed)'}

In [56]:
period = '-'.join([str(first_ex['start_year']), str(first_ex['end_year'])])
period

'1865-1904'

### Step 1 Create the bitmap

First define the bitmap keys based on the various partners.

TODO: We need to ensure we have the corrext institution mapping!

In [46]:
bitmap_keys = ["public",
"impresso",
"educational",
"researcher",
"",
"SNL",
"BNL",
"BNF",
"KBR",
"KBR",
"BL",
"ONB",
"SBB",
"SUB",
"LeTemps",
"NZZ",
"INA",
"RTS",
"BBC",
"ORF",
"Rundfunk",
"DR",
"BCUL",
"BCUF",
"Migros",
"PSNE",
"Gruyere",
"LLE",
"LCE",
"LES",
"MVS",
"FRN",
"RM",
"Syna",
"Unia",
"SWA",
"SA",
"BVCF",
"BVU",
"ArcInfo",
"Swissinfo",
"CNA", 
"SFA"
]

In [47]:
action_columns = {
        'explore': 'explore_req_status',
        'get_tr': 'get_transcript_req_status',
        'get_img': 'get_facsimile_req_status'
    }

"""action_statuses = [
    'Public Domain',
    'No restriction (all registered users allowed)',
    'Educational status at least',
    'Academic status at least'
]
status_to_check = [
    "All statuses",
    "Educational",
    "Academic and Educational",
    "No validation"
]"""

'action_statuses = [\n    \'Public Domain\',\n    \'No restriction (all registered users allowed)\',\n    \'Educational status at least\',\n    \'Academic status at least\'\n]\nstatus_to_check = [\n    "All statuses",\n    "Educational",\n    "Academic and Educational",\n    "No validation"\n]'

In [40]:
def allowed_status_to_bitmap(allowed_status, bitmap, partner_index):
    # Based on the allowed statuses, identify which indices to set to 1
    match allowed_status:
        case 'No restriction (all registered users allowed)':
            # start of bitmap: '0100'
            indices_to_set = [1]
        case 'Educational users at least OR Archive members':
            # start of bitmap: '0010'
            indices_to_set = [2, partner_index]
        case 'Educational users at least':
            # start of bitmap: '0010'
            indices_to_set = [2]
        case 'Academic users at least OR Archive members':
            # start of bitmap: '0001'
            indices_to_set = [3, partner_index]
        case 'Academic users at least':
            # start of bitmap: '0001'
            indices_to_set = [3]
        case 'Only Achive members':
            # start of bitmap: '0000'
            indices_to_set = [partner_index]
        case 'Forbidden':
            # start of bitmap: '0000'
            indices_to_set = []

    # set all selected indices to 1 (there can be 0, 1 or 2)
    for idx in indices_to_set:
        bitmap[idx] = '1'

    assert Counter(bitmap)['1'] in range(3), "The constructed bitmap should have at most two bits set to 1!"

    return bitmap

In [24]:
def entry_to_bitmaps(ar_entry, bitmap_keys=bitmap_keys, action_columns=action_columns):
    bitmaps = {
        'explore':[0]*64,
        'get_tr':[0]*64,
        'get_img':[0]*64
    }
    # if the title is public domain, only the first bit should be set to 1
    if 'Public Domain' in ar_entry["copyright_status"]:
        for bm in bitmaps.values():
            bm[0] = 1
    # if it's not public, each bitmap changes based on the various columns
    else:
        # for each type of bitmap, check the value of the column and modify the bitmap accordingly
        for bm_key, ar_key in action_columns.items():
            bitmaps[bm_key] = allowed_status_to_bitmap(ar_entry[ar_key], bitmaps[bm_key], bitmap_keys.index(ar_entry['partner_id']))
            #min_status_val = ar_entry[ar_key]
            #if 
            #bitmaps[bm_key][action_statuses.index(min_status_val)] = '1'

    str_bitmaps = {k: ''.join([str(v) for v in val]) for k,val in bitmaps.items()}
    #bytarr_bitmaps = {k: bytearray(val) for k,val in bitmaps.items()}
    byte_bitmaps = {k: bytes(val) for k,val in bitmaps.items()}
    return byte_bitmaps, str_bitmaps
        

In [16]:
def bitwise_and(a: str | bytes, b: str | bytes) -> str | bytes:
    assert len(a) == len(b), "The two bitmaps must be of the same size!"
    # first case: bytes
    if isinstance(a, bytes) and isinstance(b, bytes):
        return bytes([a[x] & b[x] for x in range(len(a))])
    # second case: strings
    elif isinstance(a, str) and isinstance(b, str):
        return [str(int(a[x]=='1' and b[x]=='1')) for x in range(len(a))]
    else:
        print("Notokay!")

In [18]:
#ex_ba_bitmaps, ex_by_bitmaps, ex_s_bitmaps = entry_to_bitmaps('BNL', first_ex)
ex_by_bitmaps, ex_s_bitmaps = entry_to_bitmaps(first_ex)
ex_by_bitmaps, ex_s_bitmaps

({'explore': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
  'get_tr': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
  'get_img': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'},
 {'explore': '1000000000000000000000000000000000000000000000000000000000000000',
  'get_tr': '1000000000000000000000000000000000000000000000000000000000000000',
  'get_img': '100

In [25]:

prec_byte_bm = None
prec_str_bm = None
for ex in fetched_ar:
    print(f"ex: {json.dumps(ex, indent=2)}")
    byte_bm, str_bm = entry_to_bitmaps(ex)
    
    print(json.dumps(str_bm, indent=2))
    #print(json.dumps(byte_bm, indent=2))
    print(byte_bm)
    # do and operation:
    if prec_byte_bm:
        and_res = bitwise_and(byte_bm['explore'], prec_byte_bm)
        
        print(f"byte_bm & prec_byte_bm: {byte_bm['explore']} &")
        print(f"                        {prec_byte_bm}")
        print(f"                      = {and_res}")
    if prec_str_bm:
        """for x in range(64):
            print(f"\nx: {x}")
            print(f"str_bm['explore'][x]: {str_bm['explore'][x]} - {type(str_bm['explore'][x])}")
            print(f"prec_str_bm[x]: {prec_str_bm[x]} - {type(prec_str_bm[x])}")
            print(f"str_bm['explore'][x] and prec_str_bm[x]: {int(str_bm['explore'][x]=='1' and prec_str_bm[x]=='1')} - {type(str_bm['explore'][x] and prec_str_bm[x])}")"""
        and_res = bitwise_and(str_bm['explore'], prec_str_bm)
        print(f"str_bm & prec_str_bm: {str_bm['explore']} &")
        print(f"                      {prec_str_bm}")
        print(f"                    = {''.join(and_res)}")

    print(f"\n\n")
    prec_byte_bm = byte_bm['get_tr']
    prec_str_bm = str_bm['get_tr']

ex: {
  "": 1,
  "partner_id": "SNL",
  "title_alias": "BLB",
  "full_title": "B\u00fcndner Landbote",
  "start_year": 1846,
  "end_year": 1847,
  "metadata_only": "yes",
  "copyright_status": "Public Domain",
  "explore_req_status": "",
  "get_transcript_req_status": "",
  "get_facsimile_req_status": ""
}
{
  "explore": "1000000000000000000000000000000000000000000000000000000000000000",
  "get_tr": "1000000000000000000000000000000000000000000000000000000000000000",
  "get_img": "1000000000000000000000000000000000000000000000000000000000000000"
}
{'explore': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'get_tr': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [26]:
str_bm['explore'][3], prec_str_bm[2], str_bm['explore'][3] and prec_str_bm[2]

('0', '0', '0')

In [27]:
print(ex_by_bitmaps['explore'])
bytes([ex_by_bitmaps['explore'][x] & ex_by_bitmaps['get_img'][x] for x in range(64)]) #, bytearray(ex_lst_bitmaps['explore'])

b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [34]:
def bitmap_bytes_to_str(bytes_bitmap: bytes) -> str:
    as_str = [str(x) for x in bytes_bitmap]
    return ''.join(as_str)

In [33]:
def bitmap_str_to_bytes(str_bitmap: str) -> bytes:
    as_int = [int(x) for x in str_bitmap]
    return bytes(as_int)

In [35]:
str_bm = bitmap_bytes_to_str(ex_by_bitmaps['explore'])

bytes_bm = bitmap_str_to_bytes(str_bm)

ex_by_bitmaps['explore'], type(ex_by_bitmaps['explore']), str_bm, type(str_bm), bytes_bm, type(bytes_bm)

(b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
 bytes,
 '1000000000000000000000000000000000000000000000000000000000000000',
 str,
 b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
 bytes)

In [37]:
def entry_to_bitmaps_2(ar_entry, bitmap_keys=bitmap_keys, action_columns=action_columns):
    bitmaps = {
        'explore':['0']*64,
        'get_tr':['0']*64,
        'get_img':['0']*64
    }
    # if the title is public domain, only the first bit should be set to 1
    if 'Public Domain' in ar_entry["copyright_status"]:
        for bm in bitmaps.values():
            bm[0] = '1'
    # if it's not public, each bitmap changes based on the various columns
    else:
        # for each type of bitmap, check the value of the column and modify the bitmap accordingly
        for bm_key, ar_key in action_columns.items():
            bitmaps[bm_key] = allowed_status_to_bitmap(ar_entry[ar_key], bitmaps[bm_key], bitmap_keys.index(ar_entry['partner_id']))
            #min_status_val = ar_entry[ar_key]
            #if 
            #bitmaps[bm_key][action_statuses.index(min_status_val)] = '1'

    str_bitmaps = {k: ''.join(val) for k,val in bitmaps.items()}
    #bytarr_bitmaps = {k: bytearray(val) for k,val in bitmaps.items()}
    byte_bitmaps = {k: bitmap_str_to_bytes(val) for k,val in bitmaps.items()}
    
    return byte_bitmaps, str_bitmaps
        

In [52]:
prec_byte_bm = None
prec_str_bm = None
for ex in fetched_ar:
    print(f"ex: {json.dumps(ex, indent=2)}")
    byte_bm, str_bm = entry_to_bitmaps_2(ex)
    
    print(json.dumps(str_bm, indent=2))
    #print(json.dumps(byte_bm, indent=2))
    print(byte_bm)
    # do and operation:
    if prec_byte_bm:
        and_res = bitwise_and(byte_bm['explore'], prec_byte_bm)
        
        print(f"byte_bm & prec_byte_bm: {byte_bm['explore']} &")
        print(f"                        {prec_byte_bm}")
        print(f"                      = {and_res}")
    if prec_str_bm:
        """for x in range(64):
            print(f"\nx: {x}")
            print(f"str_bm['explore'][x]: {str_bm['explore'][x]} - {type(str_bm['explore'][x])}")
            print(f"prec_str_bm[x]: {prec_str_bm[x]} - {type(prec_str_bm[x])}")
            print(f"str_bm['explore'][x] and prec_str_bm[x]: {int(str_bm['explore'][x]=='1' and prec_str_bm[x]=='1')} - {type(str_bm['explore'][x] and prec_str_bm[x])}")"""
        and_res = bitwise_and(str_bm['explore'], prec_str_bm)
        print(f"str_bm & prec_str_bm: {str_bm['explore']} &")
        print(f"                      {prec_str_bm}")
        print(f"                    = {''.join(and_res)}")

    print(f"\n\n")
    prec_byte_bm = byte_bm['get_tr']
    prec_str_bm = str_bm['get_tr']

ex: {
  "": 1,
  "partner_id": "Gruyere",
  "title_alias": "NGV",
  "full_title": "La nouvelle Gazette de Vaud",
  "start_year": 1865,
  "end_year": 1904,
  "metadata_only": "yes",
  "copyright_status": "Public Domain",
  "explore_req_status": "No restriction (all registered users allowed)",
  "get_transcript_req_status": "No restriction (all registered users allowed)",
  "get_facsimile_req_status": "No restriction (all registered users allowed)"
}
{
  "explore": "1000000000000000000000000000000000000000000000000000000000000000",
  "get_tr": "1000000000000000000000000000000000000000000000000000000000000000",
  "get_img": "1000000000000000000000000000000000000000000000000000000000000000"
}
{'explore': b'\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'get_tr': b'\x01\x00\x00\x00\

In [ ]:
ex_lst_bitmaps['explore']

[1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

# Corpus Access Catalogue

In [26]:
from harvesters.utils import util_slugify
import copy

In [9]:
ar_dir = "/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles"

In [11]:
ar_master_files = {}
for ar_master_file in os.listdir(ar_dir):
    if "debug" not in ar_master_file:
        partner = ar_master_file.split('.')[1]
        ar_master_files[partner] = os.path.join(ar_dir, ar_master_file)

ar_master_files

{'snl': '/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles/access_rights.snl.json',
 'bcul': '/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles/access_rights.bcul.json',
 'bnf': '/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles/access_rights.bnf.json',
 'bnl': '/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles/access_rights.bnl.json',
 'swa_fedgaz_nzz': '/Users/piconti/impresso/impresso-corpus-metadata/data/access_rights_masterfiles/access_rights.swa_fedgaz_nzz.json'}

In [25]:
util_slugify("Permitted Use")

'permitted_use'

In [37]:
CATALOGUE_RULES = {
    "rights_holder_id": {"rename_key_to": "data_partner_institution"},
    "title_alias": {"rename_key_to": "media_alias"},
    "full_title": {"rename_key_to": "media_title"},
    "start_year": {
        "rename_key_to": "time_period"
    },
    "end_year": {
        "copy_value_from_field": "end_year"
    },
    "media_type": {"rename_key_to": "media"},
    "medium": {"copy_value_from_field": "medium"},
    "content_and_metadata": {"remove_key": ""},
    "copyright_status": {
        "rename_key_to": "copyright_or_copyright_status"
    },
    "explore_req_status": {"remove_key": ""},
    "get_transcript_req_status": {"remove_key": ""},
    "get_facsimile_req_status": {"remove_key": ""},
    "allowed_use_archive_only": {
        "rename_key_to": "permitted_use"
    },
    "content_bitmaps": {"remove_key": ""},
    "rights_statement_explore": {"remove_key": ""},
    "rights_statement_get_tr": {"remove_key": ""},
    "rights_statement_get_img": {"remove_key": ""}
}

In [28]:
min_plan_cat_keys = {
    'explore': "minimum_user_plan_required_to_explore_in_the_webapp",
    "get_tr": "minimum_user_plan_required_to_export_transcripts",
    "get_img": "minimum_user_plan_required_to_export_illustration"
}

req_status_ar_keys = {
    'explore': "explore_req_status",
    "get_tr": "get_transcript_req_status",
    "get_img": "get_facsimile_req_status"
}

rights_ar_keys = {
    'explore': "rights_statement_explore",
    "get_tr": "rights_statement_get_tr",
    "get_img": "rights_statement_get_img"
}

In [54]:
def min_plan_from_rest(action_key, ar_dict, req_status_ar_key, rights_ar_key):
    # define the minimum user plan for a given action based on the required status and rights statement
    print(f"action_key: {action_key}, req_status_ar_key: {req_status_ar_key}, ar_dict[req_status_ar_key]: {ar_dict[req_status_ar_key]}")
    match ar_dict[req_status_ar_key]:
        case "No restriction (all registered users allowed)":
            if "Public Domain" in ar_dict[rights_ar_key] and action_key=='explore':
                # public domain and no restriction means gests can access it, but only explore
                return "Guest User Plan"
            else:
                # to access via the API or if the content is copyrighted, the user must be registered
                return "Basic User Plan"
        case "":
            # This is the same case as above
            if "Public Domain" in ar_dict[rights_ar_key] and action_key=='explore':
                return "Guest User Plan"
            else:
                return "Basic User Plan"
        case "Educational users at least":
            return "Academic + Student User Plan"
        case "Academic users at least":
            return "Academic User Plan"
        case "Educational users at least OR Archive members":
            # whenever archive members are allowed, we need to consider their allowed status too.
            if "Personal" in ar_dict['allowed_use_archive_only']:
                # if personal use is allowed for archive members: 
                # - basic plan can suffice if user is member of the special archive
                # - if user is not user of special archive, they need to have the Student Plan at least.
                return "Basic User Plan"
            else:
                # otherwise, any user with Student or academic plan are allowed.
                return "Student User Plan"
        case "Academic users at least OR Archive members":
            # same logic as above with one additional restriction level
            if "Personal" in ar_dict['allowed_use_archive_only']:
                return "Basic User Plan"
            elif "Educational" in ar_dict['allowed_use_archive_only']:
                return "Student User Plan"
            else:
                return "Academic User Plan"
        case "Only Archive members":
            # if only archive members are allowed, the minimum user plan is defined by the allowed uses.
            if "Personal" in ar_dict['allowed_use_archive_only']:
                return "Basic User Plan"
            elif "Educational" in ar_dict['allowed_use_archive_only']:
                return "Student User Plan"
            else:
                return "Academic User Plan"
        case "Forbidden":
            return "Not Possible"

In [32]:
def transform_value(d: dict, rules: dict[str, dict]) -> dict:
    """
    Change the value of a key in a dictionary.

    Args:
      d (dict): Input dictionary.
      rules (dict[str, dict]): Column/value transformation rules.

    Returns:
      dict: Dictionary with the updated key value.
    """
    transformed = {}

    for key in d.keys():
        slugified_key = util_slugify(key)
        transformed[slugified_key] = d[key]

    for key_with_rule, rule in rules.items():
        if "copy_value_from_field" in rule:
            val = transformed[rule["copy_value_from_field"]]
            del transformed[key_with_rule]
            transformed[key_with_rule] = val
        if "split_values_by_re" in rule:
            transformed[key_with_rule] = [
                x
                for x in re.split(
                    rule["split_values_by_re"], transformed[key_with_rule]
                )
                if x.strip()
            ]
        if "rename_key_to" in rule:
            transformed[rule["rename_key_to"]] = transformed[key_with_rule]
            del transformed[key_with_rule]
        if "remove_key" in rule:
            del transformed[key_with_rule]

    return transformed

In [55]:
def remap_keys_and_values(ar_dict, min_plan_cat_keys=min_plan_cat_keys, req_status_ar_keys=req_status_ar_keys, rights_ar_keys=rights_ar_keys):
    
    catalogue_entry = transform_value(copy.deepcopy(ar_dict), CATALOGUE_RULES)
    catalogue_entry['time_period'] = f"{catalogue_entry['time_period']}-{catalogue_entry['end_year']}"
    del catalogue_entry['end_year']

    for key, min_plan_catalogue in min_plan_cat_keys.items():
        catalogue_entry[min_plan_catalogue] = min_plan_from_rest(key, ar_dict, req_status_ar_keys[key], rights_ar_keys[key])

    return catalogue_entry


In [40]:
def to_catalogue_entry(access_rights):
    catalogue = {}
    for title, period_dict in access_rights.items():
        print(f"\n{title}:")
        catalogue[title] = {}
        for period, ar_dict in period_dict.items():
            print(f"- {period}:")
            catalogue[title][period] = remap_keys_and_values(ar_dict)
            print(f"ar_dict: {ar_dict} \ncatalogue: {catalogue[title][period]}")
            
    return catalogue

In [57]:
access_rights_contents = {}
for partner, file_path in ar_master_files.items():
    print(f"\n\n___________{partner.upper()}_________:")
    with open(file_path, "r", encoding="utf-8") as file:
        fetched_ar = json.load(file)

    tranformed_ar = to_catalogue_entry(fetched_ar)

    access_rights_contents[partner] = tranformed_ar



___________SNL_________:

BLB:
- 1846-1847:
action_key: explore, req_status_ar_key: explore_req_status, ar_dict[req_status_ar_key]: 
action_key: get_tr, req_status_ar_key: get_transcript_req_status, ar_dict[req_status_ar_key]: 
action_key: get_img, req_status_ar_key: get_facsimile_req_status, ar_dict[req_status_ar_key]: 
ar_dict: {'rights_holder_id': 'SNL', 'title_alias': 'BLB', 'full_title': 'Bündner Landbote', 'start_year': 1846, 'end_year': 1847, 'media_type': 'Newspaper', 'medium': 'print', 'content_and_metadata': 'yes', 'copyright_status': 'Public Domain', 'explore_req_status': '', 'get_transcript_req_status': '', 'get_facsimile_req_status': '', 'allowed_use_archive_only': '', 'content_bitmaps': {'explore': '1000000000000000000000000000000000000000000000000000000000000000', 'get_tr': '1000000000000000000000000000000000000000000000000000000000000000', 'get_img': '1000000000000000000000000000000000000000000000000000000000000000'}, 'rights_statement_explore': 'Public Domain (1846-1

In [58]:
access_rights_contents

{'snl': {'BLB': {'1846-1847': {'data_partner_institution': 'SNL',
    'media_alias': 'BLB',
    'media_title': 'Bündner Landbote',
    'time_period': '1846-1847',
    'media': 'Newspaper',
    'medium': 'print',
    'copyright_or_copyright_status': 'Public Domain',
    'permitted_use': '',
    'minimum_user_plan_required_to_explore_in_the_webapp': 'Guest User Plan',
    'minimum_user_plan_required_to_export_transcripts': 'Basic User Plan',
    'minimum_user_plan_required_to_export_illustration': 'Basic User Plan'}},
  'BNN': {'1885-1892': {'data_partner_institution': 'SNL',
    'media_alias': 'BNN',
    'media_title': 'Bündner Nachrichten',
    'time_period': '1885-1892',
    'media': 'Newspaper',
    'medium': 'print',
    'copyright_or_copyright_status': 'Public Domain',
    'permitted_use': '',
    'minimum_user_plan_required_to_explore_in_the_webapp': 'Guest User Plan',
    'minimum_user_plan_required_to_export_transcripts': 'Basic User Plan',
    'minimum_user_plan_required_to_exp

In [59]:
with open(os.path.join(ar_dir, "corpus_access_catalogue.json"), 'w', encoding='utf-8') as f:
    f.write(json.dumps(access_rights_contents, indent=4))